In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/moleculenet-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "applicability-domain.py"

# 模型適用範圍
用訓練分子的五近鄰距離定門檻，再以未見骨架檢查誤差。

第一次執行會由 DeepChem 下載 Lipophilicity。已有原始檔時可設定 DEEPCHEM_DATA_DIR。中間程式格寫入暫存腳本，小練習程式可接在腳本末端，再執行末兩格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

SEED = 42
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path.home() / ".cache" / "deepchem"))
cache.mkdir(parents=True, exist_ok=True)
tasks, (train, valid, test), _ = dc.molnet.load_lipo(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), save_dir=str(cache), reload=True)
assert tasks == ["exp"] and all(np.all(s.w[:, 0] > 0) for s in (train, valid, test))
print("train / valid / test:", len(train), len(valid), len(test))

def matrix(split, names):
    rows = []
    for smiles in split.ids:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"無法讀取分子 {smiles}")
        rows.append([getattr(Descriptors, name)(mol) for name in names])
    frame = pd.DataFrame(rows, columns=names)
    assert np.isfinite(frame.to_numpy()).all()
    return frame

def forest():
    return RandomForestRegressor(n_estimators=80, min_samples_leaf=2,
                                 random_state=SEED, n_jobs=1)

figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

names = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors", "NumRotatableBonds"]
scaler = StandardScaler()
X_train = scaler.fit_transform(matrix(train, names))
X_test = scaler.transform(matrix(test, names))
nn = NearestNeighbors(n_neighbors=5).fit(X_train)
train_dist = nn.kneighbors()[0].mean(axis=1)
threshold = np.quantile(train_dist, 0.95)
dist, neighbors = nn.kneighbors(X_test)
test_dist = dist.mean(axis=1)
inside = test_dist <= threshold
model = forest().fit(X_train, train.y[:, 0])
pred = model.predict(X_test)
errors = abs(test.y[:, 0] - pred)
print(f"threshold = {threshold:.4f}")
for label, mask in [("inside", inside), ("outside", ~inside)]:
    print(f"{label}: {mask.sum()}/{len(mask)}, MAE = {errors[mask].mean():.4f}")
i = int(np.argmax(test_dist))
print(f"most distant test row {i}: dist = {test_dist[i]:.4f}, predicted = {pred[i]:.3f}, observed = {test.y[i, 0]:.3f}")
print("Closest training row:", neighbors[i, 0])
fig, ax = plt.subplots(figsize=(7, 4.3), layout="constrained")
for label, mask, color in [("Inside", inside, "#236d75"), ("Outside", ~inside, "#bb5a23")]:
    ax.scatter(test_dist[mask], errors[mask], s=17, alpha=0.65, color=color,
               label=f"{label} (n={mask.sum()})", edgecolors="none")
ax.axvline(threshold, color="#333333", ls="--", lw=1.1, label="Training 95th percentile")
ax.set(xlabel="Mean distance to 5 training neighbors (scaled descriptors)",
       ylabel="Absolute error (logD units)", title="Distance and error on unseen scaffolds")
ax.legend(frameon=False)
fig.savefig(figure_dir / "10-05-applicability-domain.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "10-05-applicability-domain.png")))